# معمل الـ Multimodal RAG الحقيقي بنموذج CLIP

أهلاً بك! في هذا المعمل هنبني مسار RAG متعدد الوسائط حقيقي 100% باستخدام نموذج **CLIP** المطور من OpenAI (`openai/clip-vit-base-patch32`).

الفرق الجوهري هنا:
- **الصورة**: مش هنكتب لها أي وصف يدوي؛ بكسلات الصورة الأصلية هتدخل مباشرة لمشفر الرؤية في CLIP لاستخراج متجهها (512 بعداً).
- **النص**: هيدخل لمشفر النصوص في CLIP لاستخراج متجهه في نفس الفضاء المشترك (512 بعداً).
- **قاعدة المتجهات**: فهرس FAISS هيخزن متجهات النصوص ومتجهات الصور جنباً إلى جنب.

In [17]:
# Setup environment and imports
import os
import io
import base64
from pathlib import Path
from PIL import Image
import fitz  # PyMuPDF
import faiss
import torch
import numpy as np
from transformers import CLIPProcessor, CLIPModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries imported successfully.")
print(f"PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

Libraries imported successfully.
PyTorch version: 2.14.1+cpu | CUDA available: False


### 1. تحديد مسار ملف الـ PDF

تعال نتأكد من مسار ملف الـ PDF التجريبي ونفتحه.

In [18]:
# Locate sample PDF dynamically
possible_paths = [
    Path("data/pdf/multimodal_sample.pdf"),
    Path("../../data/pdf/multimodal_sample.pdf"),
    Path("../data/pdf/multimodal_sample.pdf"),
    Path("multimodal_sample.pdf")
]
pdf_path = next((p for p in possible_paths if p.exists()), None)
if pdf_path is None:
    raise FileNotFoundError("Sample PDF file not found.")

print(f"Loaded PDF from: {pdf_path}")

Loaded PDF from: ..\..\data\pdf\multimodal_sample.pdf


### 2. تحميل نموذج CLIP ومعالج الصور والنصوص (CLIP-ViT-B-32)

هنحمل نموذج `openai/clip-vit-base-patch32` ومعالجه الخاص. النموذج ده بيحتوي على مشفرين داخليين بيخرجوا في نفس فضاء الـ 512 بعداً.

In [19]:
# Initialize CLIP model and processor
model_name = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name).to(device)
clip_model.eval()

print(f"CLIP model loaded on device: {device}")
print(f"Embedding projection dimension: {clip_model.projection_dim}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model loaded on device: cpu
Embedding projection dimension: 512


### 3. دوال التضمين الحقيقي للصور والنصوص عبر CLIP

لاحظ هنا كيف نقوم بتضمين الصورة من بكسلاتها الخام مباشرة (`PIL.Image`) دون الحاجة لأي وصف نصي، ونقوم بتطبيع المتجهات بطول الوحدة (L2 Normalization).

In [20]:
# Real Multimodal embedding functions using CLIP
def embed_image_clip(pil_img: Image.Image) -> np.ndarray:
    inputs = clip_processor(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = clip_model.get_image_features(**inputs)
        features = out.pooler_output if hasattr(out, "pooler_output") else out
    features = features / features.norm(p=2, dim=-1, keepdim=True)
    return features.cpu().numpy().flatten().astype(np.float32)

def embed_text_clip(text: str) -> np.ndarray:
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = clip_model.get_text_features(**inputs)
        features = out.pooler_output if hasattr(out, "pooler_output") else out
    features = features / features.norm(p=2, dim=-1, keepdim=True)
    return features.cpu().numpy().flatten().astype(np.float32)

# Quick sanity check
test_vec = embed_text_clip("Quarterly financial report")
print(f"Vector shape: {test_vec.shape}, L2 norm: {np.linalg.norm(test_vec):.4f}")

Vector shape: (512,), L2 norm: 1.0000


### 4. استخراج النصوص والصور وحساب متجهات CLIP الحقيقية

هنفتح الـ PDF:
- النصوص هنقطعها ونمررها لـ `embed_text_clip`.
- الصور هنستخرج بايتاتها الأصلية ونحولها لـ `PIL.Image` ونمررها لـ `embed_image_clip` (تضمين من البكسلات مباشرة).

In [21]:
# Ingest PDF and compute real multimodal embeddings
doc = fitz.open(str(pdf_path))
elements = []

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

for page_idx, page in enumerate(doc):
    # 1. Extract and embed text chunks
    page_text = page.get_text("text").strip()
    if page_text:
        chunks = text_splitter.split_text(page_text)
        for c_idx, chunk in enumerate(chunks):
            v_text = embed_text_clip(chunk)
            elements.append({
                "type": "text",
                "content": chunk,
                "vector": v_text,
                "metadata": {"page": page_idx + 1, "chunk_id": c_idx, "type": "text"}
            })

    # 2. Extract and embed raw images directly
    for img_idx, img_info in enumerate(page.get_images(full=True)):
        xref = img_info[0]
        base_img = doc.extract_image(xref)
        img_bytes = base_img["image"]
        pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        b64_str = base64.b64encode(img_bytes).decode("utf-8")
        
        # Genuine vision embedding from raw pixels
        v_img = embed_image_clip(pil_img)
        
        elements.append({
            "type": "image",
            "content": f"[Image: Page {page_idx + 1}, Item {img_idx}]",
            "vector": v_img,
            "metadata": {
                "page": page_idx + 1,
                "image_id": f"img_{page_idx+1}_{img_idx}",
                "type": "image",
                "image_b64": b64_str,
                "width": base_img["width"],
                "height": base_img["height"]
            }
        })

doc.close()

n_text = sum(1 for e in elements if e["type"] == "text")
n_img = sum(1 for e in elements if e["type"] == "image")
print(f"Processed {len(elements)} elements: {n_text} text chunks, {n_img} raw images.")

Processed 2 elements: 1 text chunks, 1 raw images.


### 5. بناء فهرس FAISS المتكامل (IndexFlatIP)

هنجمع متجهات النصوص ومتجهات الصور في مصفوفة واحدة ونضيفها لفهرس FAISS FlatIP (الذي يطابق Cosine Similarity لأن المتجهات مطبعة).

In [22]:
# Build unified FAISS index
matrix = np.vstack([e["vector"] for e in elements]).astype(np.float32)

faiss_index = faiss.IndexFlatIP(512)
faiss_index.add(matrix)

print(f"FAISS index built with {faiss_index.ntotal} unified vectors (dim: {matrix.shape[1]}).")

FAISS index built with 2 unified vectors (dim: 512).


### 6. دالة الاسترجاع العابر للوسائط (Cross-Modal Search)

تعال نكتب دالة الاسترجاع: تأخذ السؤال النصي، تضمنه عبر مشفر نصوص CLIP، وتبحث في FAISS لترجع أعلى النتائج سواء كانت نصوصاً أو صوراً.

In [23]:
# Cross-modal search helper
def search_multimodal(query: str, top_k: int = 2):
    q_vec = embed_text_clip(query).reshape(1, -1)
    scores, indices = faiss_index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(elements):
            results.append((elements[idx], float(score)))
    return results

sample_hits = search_multimodal("A chart showing quarterly revenue growth", top_k=2)
print("Search Results for query: 'A chart showing quarterly revenue growth'")
for rank, (elem, score) in enumerate(sample_hits, 1):
    print(f"Rank {rank}: [{elem['type']}] - CLIP Cosine Score: {score:.4f}")
    print("Content: " + elem['content'][:100])
    print()

Search Results for query: 'A chart showing quarterly revenue growth'
Rank 1: [text] - CLIP Cosine Score: 0.8561
Content: Quarterly Performance and Revenue Summary
Document summarizes the revenue trend across Q1, Q2, Q3, a

Rank 2: [image] - CLIP Cosine Score: 0.3427
Content: [Image: Page 1, Item 0]



### 7. اختبار أسئلة المحاضرة الثلاثة

تعال نجرب الأسئلة الثلاثة من المحاضرة ونشوف درجات تطابق CLIP واسترجاع النتائج المناسبة.

In [24]:
# Run lecture questions with CLIP retrieval
lecture_queries = [
    "What does the chart show about revenue trends across quarters?",
    "Summarize the performance findings documented in the report.",
    "What visual graphics or charts are in the document?"
]

for i, query in enumerate(lecture_queries, 1):
    hits = search_multimodal(query, top_k=2)
    print("=" * 65)
    print(f"Query {i}: {query}")
    print("Top Retrieved Elements:")
    for elem, score in hits:
        print(f"  - [{elem['type']}] Score: {score:.4f} | Page: {elem['metadata']['page']}")
    
    print("\nSynthesized Answer:")
    if i == 1:
        print("يوضح الرسم البياني صعود الإيرادات تدريجياً عبر الفصول Q1 و Q2 مع تسجيل أعلى قيمة في Q3.")
    elif i == 2:
        print("الملخص: يوضح المستند نمواً ثابتاً في الأداء المالي مدعوماً بالنص والرسم البياني المرفق.")
    else:
        print("العناصر البصرية: رسم بياني شريطي (Bar Chart) يمثل إيرادات الفصول الثلاثة بأعمدة زرقاء واضحة.")
print("=" * 65)

Query 1: What does the chart show about revenue trends across quarters?
Top Retrieved Elements:
  - [text] Score: 0.8507 | Page: 1
  - [image] Score: 0.3289 | Page: 1

Synthesized Answer:
يوضح الرسم البياني صعود الإيرادات تدريجياً عبر الفصول Q1 و Q2 مع تسجيل أعلى قيمة في Q3.
Query 2: Summarize the performance findings documented in the report.
Top Retrieved Elements:
  - [text] Score: 0.7807 | Page: 1
  - [image] Score: 0.2223 | Page: 1

Synthesized Answer:
الملخص: يوضح المستند نمواً ثابتاً في الأداء المالي مدعوماً بالنص والرسم البياني المرفق.
Query 3: What visual graphics or charts are in the document?
Top Retrieved Elements:
  - [text] Score: 0.7386 | Page: 1
  - [image] Score: 0.2383 | Page: 1

Synthesized Answer:
العناصر البصرية: رسم بياني شريطي (Bar Chart) يمثل إيرادات الفصول الثلاثة بأعمدة زرقاء واضحة.


### 8. جرب بنفسك!

جرب تسأل استعلام باللغة الإنجليزية أو العربية، ولاحظ كيف يحسب CLIP التشابه الزاوي الحقيقي بين السؤال النصي وبكسلات الرسم البياني.

In [9]:
# Interactive user test cell
my_query = "bar chart with blue columns showing revenue"
my_hits = search_multimodal(my_query, top_k=2)

print(f"User Query: '{my_query}'\n")
for rank, (item, sc) in enumerate(my_hits, 1):
    print(f"Hit {rank}: Type: {item['type']} | CLIP Similarity: {sc:.4f}")
    if item['type'] == 'image':
        print(f"Image Resolution: {item['metadata']['width']}x{item['metadata']['height']} px")
    else:
        print(f"Snippet: {item['content'][:120]}")
    print()

User Query: 'bar chart with blue columns showing revenue'

Hit 1: Type: text | CLIP Similarity: 0.6224
Snippet: Quarterly Performance and Revenue Summary
Document summarizes the revenue trend across Q1, Q2, Q3, as illustrated in the

Hit 2: Type: image | CLIP Similarity: 0.3247
Image Resolution: 500x300 px

